In [3]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import yfinance as yf
import requests
import json
import time
import datetime
import os
import sys


In [6]:
import json
import time
import requests
import pandas as pd

def fetch_historical_data(symbol, api_token, headers={'Content-Type': 'application/json'}, output_csv='historical_data.csv'):
    """
    批量获取历史K线数据，目标获取过去20w根K线，并输出到CSV文件。
    
    参数:
        symbol: 股票/币种代码
        api_token: API令牌
        headers: 请求头
        output_csv: CSV文件保存路径，默认为 'historical_data.csv'
    """
    all_data = []
    end_timestamp = 0  # 0 表示从最新数据开始查询
    target_count = 200000  # 目标获取200,000根K线

    while len(all_data) < target_count:
        print("当前 end_timestamp:", end_timestamp, "累计获取记录数:", len(all_data))
        params = {
            "trace": "python_http_test1",
            "data": {
                "code": symbol,
                "kline_type": 1,           # 1分钟K线
                "kline_timestamp_end": end_timestamp,
                "query_kline_num": 1000,   # 每次最多1000根
                "adjust_type": 0           # 不复权
            }
        }
        query = json.dumps(params)
        query = requests.utils.quote(query)
        url = f'https://quote.alltick.io/quote-b-api/kline?token={api_token}&query={query}'

        try:
            response = requests.get(url, headers=headers)
            if response.status_code == 200:
                data = response.json()
                current_batch = data['data'].get('kline_list', [])
                if current_batch:
                    all_data.extend(current_batch)
                    # 注意：如果API返回数据为最新到最旧，建议使用 current_batch[-1]['timestamp']
                    # 这里保留原逻辑使用 current_batch[0]['timestamp']，你可根据实际情况修改
                    end_timestamp = current_batch[0]['timestamp']
                else:
                    print("没有更多数据可获取。")
                    break
            else:
                print(f"请求失败，状态码：{response.status_code}")
                break
        except Exception as e:
            print(f"请求出现异常：{e}")
            break

        # 暂停1秒，防止请求过快
        time.sleep(1)

    # 构造 DataFrame
    df = pd.DataFrame(all_data)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'].astype(int), unit='s')
        df.set_index('timestamp', inplace=True)
    for col in ['open_price', 'close_price', 'high_price', 'low_price', 'volume']:
        if col in df.columns:
            df[col] = df[col].astype(float)
    df.rename(columns={
        'open_price': 'Open',
        'close_price': 'Close',
        'high_price': 'High',
        'low_price': 'Low',
        'volume': 'Volume'
    }, inplace=True)
    df.sort_index(inplace=True)
    print("历史数据获取完毕，记录数：", len(df))
    
    # 保存为 CSV 文件
    df.to_csv(output_csv)
    print("数据已保存到", output_csv)
    
    return df

# 示例调用:
# symbol = "your_symbol_here"
# api_token = "your_api_token_here"
# headers = {"User-Agent": "your_user_agent_here"}
# df = fetch_historical_data(symbol, api_token, headers)
symbol = "GOLD"
api_token = "949751a4ea6a586f9e2805a3909d456a-c-app"
df = fetch_historical_data(symbol, api_token)


当前 end_timestamp: 0 累计获取记录数: 0
当前 end_timestamp: 1741340760 累计获取记录数: 1000
当前 end_timestamp: 1741277220 累计获取记录数: 2000
当前 end_timestamp: 1741217280 累计获取记录数: 3000
当前 end_timestamp: 1741153740 累计获取记录数: 4000
当前 end_timestamp: 1741090200 累计获取记录数: 5000
当前 end_timestamp: 1741026660 累计获取记录数: 6000
当前 end_timestamp: 1740966720 累计获取记录数: 7000
当前 end_timestamp: 1740730380 累计获取记录数: 8000
当前 end_timestamp: 1740666840 累计获取记录数: 9000
当前 end_timestamp: 1740603300 累计获取记录数: 10000
当前 end_timestamp: 1740543360 累计获取记录数: 11000
当前 end_timestamp: 1740479820 累计获取记录数: 12000
当前 end_timestamp: 1740416280 累计获取记录数: 13000
当前 end_timestamp: 1740356340 累计获取记录数: 14000
当前 end_timestamp: 1740120000 累计获取记录数: 15000
当前 end_timestamp: 1740056460 累计获取记录数: 16000
当前 end_timestamp: 1739992920 累计获取记录数: 17000
当前 end_timestamp: 1739932980 累计获取记录数: 18000
当前 end_timestamp: 1739869440 累计获取记录数: 19000
当前 end_timestamp: 1739805900 累计获取记录数: 20000
当前 end_timestamp: 1739569560 累计获取记录数: 21000
当前 end_timestamp: 1739509620 累计获取记录数: 22000
当前 end_tim